<style>
.reveal .slides section { font-size: 0.80em; padding: 8px 40px !important; box-sizing: border-box; }
.reveal .slides section img { max-height: 380px !important; width: auto; }
.reveal .slides section table { font-size: 0.85em; width: 100%; }
.reveal .slides section pre code { font-size: 0.78em; }
.reveal .slides section h1 { font-size: 1.6em; }
.reveal .slides section h2 { font-size: 1.25em; margin-bottom: 0.25em; }
</style>

<div style="text-align:center; padding-top:60px">
<h1 style="font-size:2.2em; color:#2a6496;">Deep Learning — Classificação de Radiografias Torácicas</h1>
<h3 style="color:#555;">Classificação multi-classe de imagens médicas com progressão<br>de baseline denso até ensemble de ViT pré-treinado no domínio</h3>
<br><br>
<p style="font-size:1.1em;"><b>José Sousa</b> &nbsp;|&nbsp; Data Science &nbsp;|&nbsp; ISAG &nbsp;|&nbsp; Abril 2026</p>
<p style="font-size:0.95em; color:#777;">NVIDIA Quadro RTX 3000 · TensorFlow/Keras · Python</p>
</div>

## Agenda

| # | Tema |
|---|---|
| 1 | Definição do Problema & Dataset |
| 2 | Objetivos & Métricas |
| 3 | Estratégia |
| 4 | Modelos: Baseline (4.1) → CNN (4.2) → CNN+Reg (4.3) |
| 5 | Transfer Learning: MobileNetV2 (4.4a) · EfficientNetB0 (4.4b) |
| 6 | RAD-DINO (4.5) — ViT pré-treinado no domínio |
| 7 | Pesquisa de Arquitetura — CNN Otimizada (Secção 6) |
| 8 | Avaliação: Matrizes de Confusão & Métricas por Classe |
| 9 | Ensemble Ponderado (Secção 7) · Grad-CAM · Conclusões |

## 1. Definição do Problema

**Tarefa:** Classificação multi-classe de radiografias torácicas

| Classe | Descrição | Imagens |
|---|---|---|
| 🫁 Normal | Pulmões saudáveis | 10 192 |
| 🌫 Opacidade Pulmonar | Densidades inespecíficas (pneumonia, edema…) | 6 012 |
| 🦠 COVID-19 | Opacidades bilaterais em vidro fosco por SARS-CoV-2 | 3 616 |
| 🤧 Pneumonia Viral | Outras infeções virais | 1 345 |

<br>

**Relevância:** Ferramenta de triagem automática para apoiar radiologistas — especialmente durante surtos epidémicos

**Desafio:** Elevada semelhança visual entre classes + desequilíbrio significativo (8× mais Normal do que Pneumonia Viral)

## Dataset

<div style="display:flex; gap:40px; align-items:flex-start">
<div>

**COVID-19 Radiography Database** (Rahman et al. 2021)  
<a href="https://www.kaggle.com/datasets/tawsifurrahman/covid19-radiography-database" style="color:#3498db;font-size:0.85em;">kaggle.com/datasets/tawsifurrahman/covid19-radiography-database</a>

- **21 165 imagens** no total
- PNG, **299 × 299 píxeis**, escala de cinzentos (1 canal)
- 140 imagens (0,7%) com canais pseudo-RGB → convertidas
- Divisão: **70% treino / 15% val / 15% teste** (estratificada)

</div>
<div>
<img src="pres_imgs/dataset_scale.png" style="max-height:300px">
</div>
</div>
<small><i>Dataset em contexto — as nossas 21K imagens situam-se entre o CIFAR-10 e o ImageNet</i></small>

## Amostras — Semelhança Visual Entre Classes

<img src="pres_imgs/visual_exploration.png" style="max-height:480px; display:block; margin:auto;">
<small><i>4 amostras por classe. COVID e Opacidade Pulmonar são visualmente muito semelhantes a Normal.</i></small>

## 2. Objetivos & Métricas

| Métrica | Objetivo | Justificação |
|---|---|---|
| **Acurácia no Teste** | ≥ 92% | Correção global entre todas as classes |
| **Recall COVID-19** | ≥ 0,90 | Falsos negativos = diagnósticos falhados → risco clínico |

<br>

**Racional:**
- Acurácia: bom sinal de treino, intuitivo por época
- Desequilíbrio de classes → **recall** e **F1** por classe na avaliação final
- **Recall COVID é a métrica crítica de segurança** — um modelo que falha metade dos casos COVID é clinicamente inútil
- Treino com **pesos de classe** inversamente proporcionais à frequência

## 3. Estratégia — Progressão de Modelos

```
Baseline Denso  →  CNN Personalizada  →  CNN + Regularização
       ↓
Transfer Learning (ImageNet)  →  Transfer Learning (Radiografias)
       ↓
Pesquisa de Arquitetura  →  Ensemble Ponderado
```

| Passo | Objetivo |
|---|---|
| Baseline | Estabelecer limite mínimo de desempenho |
| CNN | Provar que características espaciais são importantes |
| + Regularização | Corrigir overfitting |
| Transfer Learning | Aproveitar features pré-treinadas |
| RAD-DINO | Vantagem do conhecimento do domínio |
| Keras Tuner | Pesquisa da arquitetura ótima |
| Ensemble | Combinar a diversidade dos modelos |

## Modelo 4.1 — Baseline (Rede Densa)

<div style="display:flex;gap:40px;align-items:flex-start;margin-top:8px"><div style="flex:0 0 auto"><div style="display:inline-flex;flex-direction:column;gap:0"><div style="background:#2980b9;color:white;padding:5px 10px;border-radius:4px;width:270px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">Input (299×299×1)<span style="font-weight:normal;font-size:0.78em;display:block;margin-top:1px">89 401 píxeis</span></div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#7f8c8d;color:white;padding:5px 10px;border-radius:4px;width:270px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">Flatten<span style="font-weight:normal;font-size:0.78em;display:block;margin-top:1px">vetor 1D de 89 401 valores</span></div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#e67e22;color:white;padding:5px 10px;border-radius:4px;width:270px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">Dense (256)  ReLU<span style="font-weight:normal;font-size:0.78em;display:block;margin-top:1px">22 938 624 params</span></div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#e67e22;color:white;padding:5px 10px;border-radius:4px;width:270px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">Dense (128)  ReLU<span style="font-weight:normal;font-size:0.78em;display:block;margin-top:1px">32 896 params</span></div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#2c3e50;color:white;padding:5px 10px;border-radius:4px;width:270px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">Dense (4)  Softmax<span style="font-weight:normal;font-size:0.78em;display:block;margin-top:1px">COVID · Opac. Pulmonar · Normal · Pneumonia Viral</span></div></div></div><div style="flex:1;min-width:0"><table style="border-collapse:collapse"><tr><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em;white-space:nowrap"><b>Params treináveis</b></td><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em">~22,9M</td></tr><tr><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em;white-space:nowrap"><b>Otimizador</b></td><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em">Adam · Categorical Crossentropy</td></tr><tr><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em;white-space:nowrap"><b>Épocas</b></td><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em">20 · <b>1,8 min</b></td></tr><tr><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em;white-space:nowrap"><b>Limitação</b></td><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em">Sem noção de bordas, formas ou texturas</td></tr></table></div></div>

## Baseline — Curvas de Treino

<img src="pres_imgs/history_baseline.png" style="max-height:460px;display:block;margin:auto;">

<small><i>Val oscila e nunca converge. Val > Treino → underfitting.</i></small>


## Baseline — Comparação Acumulada

| Modelo | Acurácia Teste | Recall COVID | Params Treináveis | Tempo |
|---|---|---|---|---|
| **Baseline (4.1)**        | **77,8%** | **0,572** | **22,9M**        | **1,8 min** |


## Modelo 4.2 — CNN Personalizada

<div style="display:flex;gap:40px;align-items:flex-start;margin-top:8px"><div style="flex:0 0 auto"><div style="display:inline-flex;flex-direction:column;gap:0"><div style="background:#2980b9;color:white;padding:5px 10px;border-radius:4px;width:270px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">Input (299×299×1)</div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#27ae60;color:white;padding:5px 10px;border-radius:4px;width:270px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">Conv2D(32, 3×3) + ReLU<span style="font-weight:normal;font-size:0.78em;display:block;margin-top:1px">320 params</span></div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#16a085;color:white;padding:5px 10px;border-radius:4px;width:270px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">MaxPool(2×2)<span style="font-weight:normal;font-size:0.78em;display:block;margin-top:1px">149×149×32</span></div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#27ae60;color:white;padding:5px 10px;border-radius:4px;width:270px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">Conv2D(64, 3×3) + ReLU<span style="font-weight:normal;font-size:0.78em;display:block;margin-top:1px">18 496 params</span></div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#16a085;color:white;padding:5px 10px;border-radius:4px;width:270px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">MaxPool(2×2)<span style="font-weight:normal;font-size:0.78em;display:block;margin-top:1px">74×74×64</span></div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#27ae60;color:white;padding:5px 10px;border-radius:4px;width:270px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">Conv2D(128, 3×3) + ReLU<span style="font-weight:normal;font-size:0.78em;display:block;margin-top:1px">73 856 params</span></div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#16a085;color:white;padding:5px 10px;border-radius:4px;width:270px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">MaxPool(2×2)<span style="font-weight:normal;font-size:0.78em;display:block;margin-top:1px">37×37×128</span></div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#7f8c8d;color:white;padding:5px 10px;border-radius:4px;width:270px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">Flatten<span style="font-weight:normal;font-size:0.78em;display:block;margin-top:1px">175 232 valores</span></div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#e67e22;color:white;padding:5px 10px;border-radius:4px;width:270px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">Dense(128) + ReLU<span style="font-weight:normal;font-size:0.78em;display:block;margin-top:1px">22 429 824 params</span></div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#2c3e50;color:white;padding:5px 10px;border-radius:4px;width:270px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">Dense(4) Softmax</div></div></div><div style="flex:1;min-width:0"><table style="border-collapse:collapse"><tr><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em;white-space:nowrap"><b>Params treináveis</b></td><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em">~22,5M</td></tr><tr><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em;white-space:nowrap"><b>Melhoria</b></td><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em">Hierarquia espacial (bordas→formas→padrões)</td></tr><tr><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em;white-space:nowrap"><b>Épocas</b></td><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em">15 · <b>11,4 min</b></td></tr><tr><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em;white-space:nowrap"><b>Filtros</b></td><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em">3 blocos conv com profundidade crescente 32→64→128</td></tr></table></div></div>

## CNN — Curvas de Treino

⚠ Overfitting severo — gap treino/val de 13%

<img src="pres_imgs/history_cnn.png" style="max-height:460px;display:block;margin:auto;">

<small><i>Treino → 99%, Validação → 86%. Overfitting claro a partir da época 5.</i></small>


## CNN — Comparação Acumulada

| Modelo | Acurácia Teste | Recall COVID | Params Treináveis | Tempo |
|---|---|---|---|---|
| Baseline (4.1)        | 77,8% | 0,572 | 22,9M        | 1,8 min |
| **CNN (4.2)**             | **86,2%** | **0,923** | **22,5M**        | **11,4 min** |


## Modelo 4.3 — CNN + Regularização

<div style="display:flex;gap:40px;align-items:flex-start;margin-top:8px"><div style="flex:0 0 auto"><div style="display:inline-flex;flex-direction:column;gap:0"><div style="background:#2980b9;color:white;padding:5px 10px;border-radius:4px;width:270px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">Input (299×299×1)</div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#27ae60;color:white;padding:5px 10px;border-radius:4px;width:270px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">Conv2D(32) + ReLU + L2</div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#16a085;color:white;padding:5px 10px;border-radius:4px;width:270px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">MaxPool(2×2)</div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#27ae60;color:white;padding:5px 10px;border-radius:4px;width:270px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">Conv2D(64) + ReLU + L2</div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#16a085;color:white;padding:5px 10px;border-radius:4px;width:270px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">MaxPool(2×2)</div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#27ae60;color:white;padding:5px 10px;border-radius:4px;width:270px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">Conv2D(128) + ReLU + L2</div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#16a085;color:white;padding:5px 10px;border-radius:4px;width:270px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">MaxPool(2×2)</div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#7f8c8d;color:white;padding:5px 10px;border-radius:4px;width:270px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">Flatten</div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#e67e22;color:white;padding:5px 10px;border-radius:4px;width:270px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">Dense(128) + ReLU + L2</div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#e74c3c;color:white;padding:5px 10px;border-radius:4px;width:270px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">Dropout(0.5)<span style="font-weight:normal;font-size:0.78em;display:block;margin-top:1px">desativa 50% dos neurónios</span></div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#2c3e50;color:white;padding:5px 10px;border-radius:4px;width:270px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">Dense(4) Softmax</div></div></div><div style="flex:1;min-width:0"><table style="border-collapse:collapse"><tr><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em;white-space:nowrap"><b>Adição face à 4.2</b></td><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em">L2 em todos os Conv2D + Dense(128)</td></tr><tr><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em;white-space:nowrap"><b>Adição face à 4.2</b></td><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em">Dropout(0.5) após Dense(128)</td></tr><tr><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em;white-space:nowrap"><b>Gap treino/val</b></td><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em">~0% (era 13% na 4.2)</td></tr><tr><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em;white-space:nowrap"><b>Épocas</b></td><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em">40 · <b>31,1 min</b></td></tr></table></div></div>

## CNN + Regularização — Curvas de Treino

Recall COVID **0,952** — melhor recall individual até agora. Sem overfitting.

<img src="pres_imgs/history_cnn_reg.png" style="max-height:460px;display:block;margin:auto;">

<small><i>Treino e validação mantêm-se próximos ao longo das 40 épocas.</i></small>


## CNN + Regularização — Comparação Acumulada

| Modelo | Acurácia Teste | Recall COVID | Params Treináveis | Tempo |
|---|---|---|---|---|
| Baseline (4.1)        | 77,8% | 0,572 | 22,9M        | 1,8 min |
| CNN (4.2)             | 86,2% | 0,923 | 22,5M        | 11,4 min |
| **CNN + Reg (4.3)**       | **85,6%** | **0,952** | **22,5M**        | **31,1 min** |


## 4.4 Transfer Learning — Pré-treinado no ImageNet

<div style="display:flex;gap:40px;align-items:flex-start;margin-top:8px"><div style="flex:0 0 auto"><div style="display:inline-flex;flex-direction:column;gap:0"><div style="background:#2980b9;color:white;padding:5px 10px;border-radius:4px;width:290px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">Input (224×224×3)</div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#6c3483;color:white;padding:5px 10px;border-radius:4px;width:290px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">Backbone ImageNet  ❄ CONGELADA<span style="font-weight:normal;font-size:0.78em;display:block;margin-top:1px">MobileNetV2: 2,3M  |  EfficientNetB0: 4,1M</span></div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#8e44ad;color:white;padding:5px 10px;border-radius:4px;width:290px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">GlobalAveragePooling2D</div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#e67e22;color:white;padding:5px 10px;border-radius:4px;width:290px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">Dense(128) + ReLU<span style="font-weight:normal;font-size:0.78em;display:block;margin-top:1px">164K params treináveis</span></div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#e74c3c;color:white;padding:5px 10px;border-radius:4px;width:290px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">Dropout(0.5)</div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#2c3e50;color:white;padding:5px 10px;border-radius:4px;width:290px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">Dense(4) Softmax</div></div></div><div style="flex:1;min-width:0"><table style="border-collapse:collapse"><tr><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em;white-space:nowrap"><b>MobileNetV2</b></td><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em">Convoluções separáveis em profundidade · 2,3M frozen</td></tr><tr><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em;white-space:nowrap"><b>EfficientNetB0</b></td><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em">Escalamento composto (profundidade + largura + resolução) · 4,1M frozen</td></tr><tr><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em;white-space:nowrap"><b>Treináveis</b></td><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em">164K (vs 22,5M das CNNs scratch)</td></tr><tr><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em;white-space:nowrap"><b>Vantagem</b></td><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em">Features de 1,2M imagens ImageNet sem custo de treino</td></tr></table></div></div>

## MobileNetV2 — Curvas de Treino

Tempo de treino: **6,8 min** — o mais rápido entre os modelos com boa performance

<img src="pres_imgs/history_mobilenet.png" style="max-height:460px;display:block;margin:auto;">

<small><i>Arranque a frio forte (~83% na época 1) graças às features ImageNet.</i></small>


## MobileNetV2 — Comparação Acumulada

| Modelo | Acurácia Teste | Recall COVID | Params Treináveis | Tempo |
|---|---|---|---|---|
| Baseline (4.1)        | 77,8% | 0,572 | 22,9M        | 1,8 min |
| CNN (4.2)             | 86,2% | 0,923 | 22,5M        | 11,4 min |
| CNN + Reg (4.3)       | 85,6% | 0,952 | 22,5M        | 31,1 min |
| **MobileNetV2 (4.4a)**    | **88,6%** | **0,913** | **164K+2,3M ❄**  | **6,8 min** |


## EfficientNetB0 — Curvas de Treino

🎯 **Objetivo de acurácia atingido** (92,4% ≥ 92%) · Recall COVID 0,950 ≥ 0,90

<img src="pres_imgs/history_efficientnet.png" style="max-height:460px;display:block;margin:auto;">

<small><i>Escalamento composto confere vantagem clara sobre o MobileNetV2.</i></small>


## EfficientNetB0 — Comparação Acumulada

| Modelo | Acurácia Teste | Recall COVID | Params Treináveis | Tempo |
|---|---|---|---|---|
| Baseline (4.1)        | 77,8% | 0,572 | 22,9M        | 1,8 min |
| CNN (4.2)             | 86,2% | 0,923 | 22,5M        | 11,4 min |
| CNN + Reg (4.3)       | 85,6% | 0,952 | 22,5M        | 31,1 min |
| MobileNetV2 (4.4a)    | 88,6% | 0,913 | 164K+2,3M ❄  | 6,8 min |
| **EfficientNetB0 (4.4b)** | **92,4%** | **0,950** | **164K+4,1M ❄**  | **11,9 min** |


## Modelo 4.5 — RAD-DINO (ViT Pré-treinado no Domínio)

<div style="display:flex;gap:40px;align-items:flex-start;margin-top:8px"><div style="flex:0 0 auto"><div style="display:inline-flex;flex-direction:column;gap:0"><div style="background:#2980b9;color:white;padding:5px 10px;border-radius:4px;width:300px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">Input (518×518)</div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#6c3483;color:white;padding:5px 10px;border-radius:4px;width:300px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">RAD-DINO ViT  ❄ CONGELADO<span style="font-weight:normal;font-size:0.78em;display:block;margin-top:1px">86M params · chest X-rays · DINO auto-supervisionado</span></div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#8e44ad;color:white;padding:5px 10px;border-radius:4px;width:300px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">CLS token → 768 dim<span style="font-weight:normal;font-size:0.78em;display:block;margin-top:1px">extraído 1× e guardado em disco</span></div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#e67e22;color:white;padding:5px 10px;border-radius:4px;width:300px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">Dense(256) + ReLU</div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#e74c3c;color:white;padding:5px 10px;border-radius:4px;width:300px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">Dropout(0.3)</div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#2c3e50;color:white;padding:5px 10px;border-radius:4px;width:300px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">Dense(4) Softmax<span style="font-weight:normal;font-size:0.78em;display:block;margin-top:1px">198K params treináveis</span></div></div></div><div style="flex:1;min-width:0"><table style="border-collapse:collapse"><tr><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em;white-space:nowrap"><b>Backbone congelada</b></td><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em">86M params · pré-treinada em radiografias torácicas</td></tr><tr><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em;white-space:nowrap"><b>Classificador treinável</b></td><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em"><b>198K params</b></td></tr><tr><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em;white-space:nowrap"><b>Pipeline</b></td><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em">Fase 1: extração (1×, guardar disco) → Fase 2: treino classificador</td></tr><tr></table></div></div>

## RAD-DINO — Curvas de Treino

198K params treináveis superam CNN de 22,5M em **8,8%**

<img src="pres_imgs/history_rad_dino.png" style="max-height:460px;display:block;margin:auto;">

<small><i>Arranque a frio a ~92% (época 1). Converge sem overfitting.</i></small>


## RAD-DINO — Comparação Acumulada

| Modelo | Acurácia Teste | Recall COVID | Params Treináveis | Tempo |
|---|---|---|---|---|
| Baseline (4.1)        | 77,8% | 0,572 | 22,9M        | 1,8 min |
| CNN (4.2)             | 86,2% | 0,923 | 22,5M        | 11,4 min |
| CNN + Reg (4.3)       | 85,6% | 0,952 | 22,5M        | 31,1 min |
| MobileNetV2 (4.4a)    | 88,6% | 0,913 | 164K+2,3M ❄  | 6,8 min |
| EfficientNetB0 (4.4b) | 92,4% | 0,950 | 164K+4,1M ❄  | 11,9 min |
| **RAD-DINO (4.5)**        | **94,4%** | **0,970** | **198K+86M ❄**   | **~31 min** |


## 4.6 Comparação de Modelos

<img src="pres_imgs/model_comparison.png" style="max-height:480px; display:block; margin:auto;">
<small><i>RAD-DINO (198K treináveis) supera CNN+Reg (22,5M treináveis) em 8,8%. Conhecimento do domínio > número de parâmetros.</i></small>

## Secção 6 — Pesquisa de Arquitetura: CNN Otimizada a 75×75

<div style="display:flex;gap:40px;align-items:flex-start;margin-top:8px"><div style="flex:0 0 auto"><div style="display:inline-flex;flex-direction:column;gap:0"><div style="background:#2980b9;color:white;padding:5px 10px;border-radius:4px;width:300px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">Input (75×75×1) + Augmentation<span style="font-weight:normal;font-size:0.78em;display:block;margin-top:1px">flip · rotação · zoom</span></div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#27ae60;color:white;padding:5px 10px;border-radius:4px;width:300px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">Conv2D(32) + BatchNorm + ReLU</div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#16a085;color:white;padding:5px 10px;border-radius:4px;width:300px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">MaxPool(2×2)</div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#27ae60;color:white;padding:5px 10px;border-radius:4px;width:300px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">Conv2D(64) + BatchNorm + ReLU</div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#16a085;color:white;padding:5px 10px;border-radius:4px;width:300px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">MaxPool(2×2)</div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#27ae60;color:white;padding:5px 10px;border-radius:4px;width:300px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">Conv2D(128) + BatchNorm + ReLU</div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#16a085;color:white;padding:5px 10px;border-radius:4px;width:300px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">MaxPool(2×2)</div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#7f8c8d;color:white;padding:5px 10px;border-radius:4px;width:300px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">Flatten</div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#e67e22;color:white;padding:5px 10px;border-radius:4px;width:300px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">Dense(256) + ReLU</div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#e74c3c;color:white;padding:5px 10px;border-radius:4px;width:300px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">Dropout(0.2)</div><div style="text-align:center;color:#888;font-size:0.85em;line-height:1.1">▼</div><div style="background:#2c3e50;color:white;padding:5px 10px;border-radius:4px;width:300px;text-align:center;font-family:monospace;font-size:0.78em;font-weight:bold;margin:0 auto">Dense(4) Softmax<span style="font-weight:normal;font-size:0.78em;display:block;margin-top:1px">~500K params · encontrada pelo Keras Tuner Hyperband</span></div></div></div><div style="flex:1;min-width:0"><table style="border-collapse:collapse"><tr><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em;white-space:nowrap"><b>Algoritmo</b></td><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em">Keras Tuner · Hyperband</td></tr><tr><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em;white-space:nowrap"><b>Subset de pesquisa</b></td><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em">30% do treino (velocidade)</td></tr><tr><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em;white-space:nowrap"><b>Pesquisado</b></td><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em">n_blocks · filtros · dropout · L2 · learning rate</td></tr><tr><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em;white-space:nowrap"><b>Resolução</b></td><td style="padding:5px 10px;border-bottom:1px solid #333;font-size:0.8em">75×75 — 16× menor que as CNNs anteriores</td></tr></table></div></div>

## Secção 6 — Keras Tuner: O que foi pesquisado?

<div style="display:flex;gap:40px;align-items:flex-start;margin-top:8px"><div style="flex:1;min-width:0">

**Pesquisado pelo Hyperband**

| Hiperparâmetro | Espaço de pesquisa | Melhor valor |
|---|---|---|
| Nº de blocos conv | 1 – 4 | **3** |
| Filtros por bloco | 32, 64, 128, 256 | **32 → 64 → 128** |
| Taxa de dropout | 0,0 – 0,5 | **0,2** |
| Regularização L2 | 0 – 1×10⁻² | **sem L2** |
| Learning rate | 1×10⁻⁴ – 1×10⁻² | **~1×10⁻³** |

</div><div style="flex:1;min-width:0">

**Fixo (não pesquisado)**

| Componente | Valor fixo |
|---|---|
| Resolução de entrada | 75×75 |
| Otimizador | Adam |
| Função de perda | Categorical Crossentropy |
| Ativação conv | ReLU |
| Pooling por bloco | MaxPool 2×2 |
| BatchNormalization | Ativo em todos os blocos |
| Nº de classes | 4 |

</div></div>

## CNN Otimizada — Comparação Acumulada

| Modelo | Acurácia Teste | Recall COVID | Params Treináveis | Tempo |
|---|---|---|---|---|
| Baseline (4.1)        | 77,8% | 0,572 | 22,9M        | 1,8 min |
| CNN (4.2)             | 86,2% | 0,923 | 22,5M        | 11,4 min |
| CNN + Reg (4.3)       | 85,6% | 0,952 | 22,5M        | 31,1 min |
| MobileNetV2 (4.4a)    | 88,6% | 0,913 | 164K+2,3M ❄  | 6,8 min |
| EfficientNetB0 (4.4b) | 92,4% | 0,950 | 164K+4,1M ❄  | 11,9 min |
| RAD-DINO (4.5)        | 94,4% | 0,970 | 198K+86M ❄   | ~31 min |
| **CNN Otimizada (6)**     | **88,6%** | **0,821** | **~500K**        | **~25 min** |


## Secção 5 — Avaliação no Conjunto de Teste

**O que é o test set e porquê é importante?**

Durante o desenvolvimento dos modelos usámos apenas dois conjuntos:

| Conjunto | Tamanho | Papel |
|---|---|---|
| **Treino** | 70% (14 815 imagens) | Ajustar os pesos do modelo |
| **Validação** | 15% (3 175 imagens) | Escolher hiperparâmetros, detetar overfitting |
| **Teste** | 15% (3 175 imagens) | ⚠ Nunca visto durante o desenvolvimento |

<br>

O modelo **nunca viu o test set** — nem para treinar, nem para tomar decisões de arquitetura.

É a única estimativa honesta do desempenho no mundo real. As curvas de treino dizem-nos que o modelo *aprendeu* — o test set diz-nos se *generalizou*.

## Avaliação — Matrizes de Confusão (Conjunto de Teste)

<img src="pres_imgs/confusion_matrices.png" style="max-height:480px; display:block; margin:auto;">
<small><i>Diagonal = previsões corretas. Opacidade Pulmonar é o ponto cego universal — confundida com Normal.</i></small>

## Recall por Classe em Todos os Modelos

<img src="pres_imgs/per_class_metrics.png" style="max-height:360px; display:block; margin:auto;">

**Observações:**
- **Opacidade Pulmonar** é o ponto cego universal (confundida com Normal — clinicamente esperado)
- **Recall COVID Baseline (0,572):** falha quase metade dos casos — inaceitável
- **Recall COVID RAD-DINO (0,970):** falha apenas 16 dos 542 casos

## Secção 7 — Ensemble Ponderado

**Motivação:** Modelos diferentes cometem erros diferentes. Combiná-los cancela erros.

| Modelo | Acurácia Teste | Recall COVID | Peso |
|---|---|---|---|
| RAD-DINO | 94,4% | 0,970 | 0,261 |
| EfficientNetB0 | 92,4% | 0,950 | 0,255 |
| MobileNetV2 | 88,6% | 0,913 | 0,245 |
| **CNN + Reg** | 85,6% | **0,952** | 0,239 |

**Método:** Média ponderada dos vetores softmax → argmax → previsão final

## Ensemble — Matriz de Confusão

<img src="pres_imgs/ensemble_confusion.png" style="max-height:460px;display:block;margin:auto;">



## Ensemble Ponderado — Comparação Acumulada

| Modelo | Acurácia Teste | Recall COVID | Params Treináveis | Tempo |
|---|---|---|---|---|
| Baseline (4.1)        | 77,8% | 0,572 | 22,9M        | 1,8 min |
| CNN (4.2)             | 86,2% | 0,923 | 22,5M        | 11,4 min |
| CNN + Reg (4.3)       | 85,6% | 0,952 | 22,5M        | 31,1 min |
| MobileNetV2 (4.4a)    | 88,6% | 0,913 | 164K+2,3M ❄  | 6,8 min |
| EfficientNetB0 (4.4b) | 92,4% | 0,950 | 164K+4,1M ❄  | 11,9 min |
| RAD-DINO (4.5)        | 94,4% | 0,970 | 198K+86M ❄   | ~31 min |
| CNN Otimizada (6)     | 88,6% | 0,821 | ~500K        | ~25 min |
| **Ensemble (7)**          | **94,9%** | **0,987** | — | — |


## Secção 8 — Grad-CAM: O que o EfficientNetB0 Vê?

**Gradient-weighted Class Activation Mapping** — mapa de calor que destaca os píxeis mais influentes na decisão do modelo.

**Porquê isto é importante:** Alta acurácia ≠ raciocínio correto. O modelo pode explorar artefactos irrelevantes (*shortcut learning*).

<div style="background:#1e3a52;border-left:4px solid #3498db;padding:9px 16px;border-radius:0 4px 4px 0;margin-top:14px;font-size:0.88em;color:#cce4f7;">Queremos saber se o modelo olha para os pulmões — ou para as etiquetas do equipamento.</div>

## Grad-CAM — O que o Modelo Vê?

<img src="pres_imgs/grad_cam.png" style="max-height:580px;display:block;margin:auto;">

<small><i>EfficientNetB0 · uma amostra corretamente classificada por classe</i></small>

## Grad-CAM — Análise por Classe

| Classe | Para onde o modelo olha | Veredicto clínico |
|---|---|---|
| **COVID-19** | Ambos os campos pulmonares (opacidades bilaterais) | ✅ Clinicamente correto |
| **Opacidade Pulmonar** | Anotações de texto no canto inferior direito | ❌ Shortcut learning |
| **Normal** | Tórax superior / silhueta cardíaca | ⚠ Questionável |
| **Pneumonia Viral** | Artefacto de borda no canto superior direito | ❌ Shortcut learning |

**Conclusão:** apenas o COVID mostra atenção clinicamente relevante. Alta acurácia ≠ raciocínio correto — 3 de 4 classes exploram artefactos, não patologia pulmonar.

## Secção 9 — Resumo da Jornada

| # | Modelo | Acurácia Teste | Recall COVID | Lição Principal |
|---|---|---|---|---|
| 4.1 | Baseline        | 77,8% | 0,572 | Features globais são insuficientes |
| 4.2 | CNN | 86,2% | 0,923 | Hierarquia espacial é importante |
| 4.3 | CNN + Regularização | 85,6% | **0,952** | Regularização melhora recall, nem sempre acurácia |
| 4.4a | MobileNetV2    | 88,6% | 0,913 | Transfer learning supera treino do zero |
| 4.4b | EfficientNetB0 | 92,4% | 0,950 | Escalamento composto vale a pena |
| 4.5 | RAD-DINO | 94,4% | 0,970 | Pré-treino no domínio é decisivo |
| 6   | CNN Otimizada (75×75) | 88,6% | 0,821 | Pesquisa de arquitetura funciona |
| **7** | **Ensemble Ponderado** | **94,9%** | **0,987** | Diversidade > força individual |

## Conclusões Principais

1. **O pré-treino no domínio é decisivo**
   RAD-DINO (radiografias) supera EfficientNetB0 (ImageNet) apesar de arquiteturas semelhantes

2. **Mais parâmetros ≠ melhores resultados**
   198K params (RAD-DINO) superam 22,5M params (CNN) em **8,8%**

3. **Ensembles funcionam pela diversidade**
   O modelo mais fraco (CNN+Reg, 85,6%) eleva o recall COVID para **0,987**

4. **Shortcut learning é real e perigoso**
   Grad-CAM: 3 de 4 classes ativam em artefactos, não em patologia pulmonar

<div style="text-align:center; padding-top:40px">
<h2>Objetivos Atingidos ✓</h2>
<br>
<table style="margin:auto; font-size:1.3em; border-collapse:collapse">
<tr>
  <th style="padding:12px 24px; border-bottom:2px solid #ccc">Métrica</th>
  <th style="padding:12px 24px; border-bottom:2px solid #ccc">Objetivo</th>
  <th style="padding:12px 24px; border-bottom:2px solid #ccc">Melhor Resultado</th>
  <th style="padding:12px 24px; border-bottom:2px solid #ccc">Modelo</th>
</tr>
<tr>
  <td style="padding:10px 24px"><b>Acurácia no Teste</b></td>
  <td style="padding:10px 24px">≥ 92%</td>
  <td style="padding:10px 24px; color:green; font-weight:bold">94,9%</td>
  <td style="padding:10px 24px">Ensemble Ponderado</td>
</tr>
<tr>
  <td style="padding:10px 24px"><b>Recall COVID-19</b></td>
  <td style="padding:10px 24px">≥ 0,90</td>
  <td style="padding:10px 24px; color:green; font-weight:bold">0,987</td>
  <td style="padding:10px 24px">Ensemble Ponderado</td>
</tr>
</table>
<br>
<p style="font-size:1em; color:#555;">Ambos os objetivos atingidos — primeiro pelo EfficientNetB0, depois superados pelo ensemble.</p>
</div>

## Para Produção — Limitações deste Trabalho

O modelo foi treinado num único dataset público, com uma só fonte e equipamento.
O Grad-CAM revelou shortcut learning em 3 de 4 classes.

| Limitação | Impacto |
|---|---|
| **Dataset único** | COVID-19 Radiography Database — sem diversidade de equipamentos ou populações |
| **Shortcut learning** | 3/4 classes ativam em artefactos (bordas, anotações de texto), não em patologia |
| **Sem validação externa** | Desempenho não testado noutros hospitais ou populações |
| **Desequilíbrio de classes** | 8× mais Normal do que Pneumonia Viral — pesos de classe mitigam, não eliminam |

## O que falta para uso hospitalar?

| Requisito | Detalhe |
|---|---|
| **Validação externa** | Dataset de outro hospital, outra máquina, outra população |
| **Estudos clínicos** | Comparação com radiologistas em contexto real |
| **Regulatório (EU)** | MDR 2017/745 — classificado como *Software as Medical Device* (SaMD) |
| **Normas técnicas** | ISO 13485 · ISO 14971 · IEC 62304 |
| **Corrigir shortcut learning** | Retreino com imagens sem anotações e artefactos de borda |

**Thresholds clínicos por aplicação:**

| Aplicação | Requisito |
|---|---|
| Triagem *(screening)* | Recall ≥ 95–99% |
| Diagnóstico assistido | Recall alto + Precision razoável |

## O que pode ser útil

❌ **Não deve fazer:** diagnóstico autónomo · decisão final sem médico

✅ **Aplicações adequadas:**
- **Triagem automática** — de 1000 RX/dia, identificar suspeitos para o médico priorizar
- **Priorização de exames** — casos críticos tratados mais rápido
- ***Second opinion*** — apoio à decisão, não substituição

**Requisitos para deployment:**

| | |
|---|---|
| **Explicabilidade** | Grad-CAM obrigatório — os nossos resultados mostram que 3/4 classes não são clinicamente fiáveis sem interpretação |
| **Segurança** | Dados anonimizados (RGPD) · logging · versioning do modelo |
| **Calibração** | Probabilidades calibradas — tão importante como a acurácia |

<div style="text-align:center; padding-top:80px">
<h1 style="font-size:2.6em; color:#2a6496;">Obrigado</h1>
<p style="font-size:1.1em; color:#555; margin-top:32px;"><b>José Sousa</b> &nbsp;|&nbsp; Data Science &nbsp;|&nbsp; ISAG &nbsp;|&nbsp; Abril 2026</p>
<p style="font-size:0.9em; color:#888; margin-top:40px;">Código e notebook disponíveis no repositório do projeto</p>
</div>